In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip /content/drive/MyDrive/BootGenAI/titanic_dataset.zip

Archive:  /content/drive/MyDrive/BootGenAI/titanic_dataset.zip
   creating: titanic dataset/
  inflating: titanic dataset/gender_submission.csv  
  inflating: titanic dataset/test.csv  
  inflating: titanic dataset/train.csv  


In [3]:
import pandas as pd

df = pd.read_csv("/content/titanic dataset/train.csv")
# Check initial shape
print("Initial shape:", df.shape)

Initial shape: (891, 12)


In [4]:
#identify the numerical column
df[df.select_dtypes(include=['number']).columns].head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
0,1,0,3,22.0,1,0,7.2500
1,2,1,1,38.0,1,0,71.2833
2,3,1,3,26.0,0,0,7.9250
3,4,1,1,35.0,1,0,53.1000
4,5,0,3,35.0,0,0,8.0500


In [7]:
#Apply Standardization(Z-score scalling) => Gaussian distribution
from sklearn.preprocessing import StandardScaler

# Colonnes numériques à standardiser
cols = ['Age', 'Fare'] #Car il on des valeur grandes mais pas dominant

# Créer le scaler
scaler = StandardScaler()

df[['Age_scale','Fare_scale']] = scaler.fit_transform(df[cols])

print(df[['Age_scale','Fare_scale']].head())


   Age_scale  Fare_scale
0  -0.530377   -0.502445
1   0.571831    0.786845
2  -0.254825   -0.488854
3   0.365167    0.420730
4   0.365167   -0.486337


In [12]:
#Apply Mini-Max normalization => bounded ranges.
from sklearn.preprocessing import MinMaxScaler

normalizer = MinMaxScaler()
df[['Age_normalized','Fare_normalized']] = normalizer.fit_transform(df[['Age','Fare']])

print(df[['Age_normalized','Fare_normalized']].head())


   Age_normalized  Fare_normalized
0        0.271174         0.014151
1        0.472229         0.139136
2        0.321438         0.015469
3        0.434531         0.103644
4        0.434531         0.015713


### The effect of scaling and normalization on model performance.

* **Problem:** Age and Fare have large values → they can dominate magnitude‑sensitive models (like KNN, SVM).

* **Normalization (Min‑Max):** brings all values into the range [0,1] → balances features.

* **Standardization (Z‑score):** centers values around 0 with standard deviation = 1 → makes them comparable.

* **Overall effect:** prevents domination by large features, improves convergence, and stabilizes model performance.

In [18]:
#create a new features
df['Family_size'] = df['SibSp'] + df['Parch']+1 #The +1 counts the passenger themselves.

In [20]:
#Is Alone
df['Is_alone'] = df["Family_size"].apply(lambda x: 1 if  x==1  else 0)

print(df[['Family_size','Is_alone']].head())

   Family_size  Is_alone
0            2         0
1            2         0
2            1         1
3            2         0
4            1         1


In [26]:
#Explore the relationship
#Being alone or having family influenced survival chances.

print(df[['Family_size','Is_alone','Survived']].groupby(['Family_size','Is_alone']).mean())

                      Survived
Family_size Is_alone          
1           1         0.303538
2           0         0.552795
3           0         0.578431
4           0         0.724138
5           0         0.200000
6           0         0.136364
7           0         0.333333
8           0         0.000000
11          0         0.000000
